# 모델 개발 전 데이터 셋 정리 과정
1. 중복된 화합물 개수 찾아내기
2. 중복된 화합물 중 label 정보가 일치하지 않는 화합물 찾기
3. smiles 코드가 없는 물질 제거하기
4. 최종 데이터 확인 (label의 밸런스 확인)

In [59]:
import pandas as pd

df = pd.read_excel('skin_irritation.xlsx', sheet_name=2)

In [60]:
df.columns

Index(['Record_ID', 'Data_Type', 'Formulation_ID', 'Formulation_Name',
       'Chemical_Name', 'CASRN', 'DTXSID', 'Percent_Active_Ingredient',
       'Concentration', 'Concentration_Units', 'Mixture', 'Species',
       'Reported_Strain', 'Strain', 'Sex', 'Assay', 'Endpoint', 'Response',
       'Response_Unit', 'Reference', 'SMILES', 'Preferred_Name', 'Synonyms',
       'URL_CompTox', 'URL_CEBS'],
      dtype='str')

In [61]:
df_selec = df[(df['Mixture']=='Chemical') & (df['Species']=='Human')] #Mixture 컬럼이 Chemical, Species 컬럼은 Human인 경우 선택.

In [62]:
df_selec['Endpoint'].value_counts()

Endpoint
Qualitative classification    90
Positive reaction             90
Name: count, dtype: int64

In [63]:
df_class = df_selec[df_selec['Endpoint']=='Qualitative classification']

In [64]:
df_react = df_selec[df_selec['Endpoint']=='Positive reaction']

In [65]:
df_class['Response'].value_counts()

Response
Not classified                      65
Irritant                            14
Not classified/Possible irritant     8
Irritant/Possible corrosive          3
Name: count, dtype: int64

In [66]:
# Qualitatie classification 정리
df_class['label']=0  #label을 0으로 먼저 초기화하고,,,
df_class.loc[df_class['Response']!='Not classified','label']=1  #Not classified가 아니라면 (!=) label을 1로 설정.

In [67]:
# Positive reaction 정리
df_react['label']=0
df_react.loc[df_react['Response']!='0','label']=1 #Respons 값이 '0'이 아니라면 (!=) label을 1로 설정.

# Quiz1
왜 label을 정의할 때, Response 컬럼을 굳이 numerical value로 바꾸지 않아도 될까요?

단순 조건 판별 목적 라벨링은 데이터의 수학적 계산이나 크기 비교가 목적이 아니다. 단지 '0' 또는 'Not classified'와 같은 특정 문자열과 일치하는지만 확인하여 0과 1로 나누는 단순 이진 분류 작업이므로 문자열 조건식만으로도 충분하다.

In [68]:
df_total = pd.concat([df_class, df_react])
df_total['label'].value_counts()

label
0    102
1     78
Name: count, dtype: int64

# 데이터 중복 문제
1. 180개 데이터가 있지만, 화합물 개수는 중복되어 있을 수 있으니 확인 필요.
2. 중복된 화합물 중에서 label이 불일치 하는 경우가 있는지 확인 필요.

In [69]:
df_total['Chemical_Name']

0                  Alcohol ethoxylate C12-15/E5 phosphate
1           C12-13 beta-branched primary alco hol sulfate
2       C12-13 beta-branched primary alco hol sulfate/...
5                   N,N-Dimethyl-N-dodecyl amino- betaine
6       Bis[(1-Methylimidazol)-(2- ethyl- hexanoate)],...
                              ...                        
1850                                      Triethanolamine
1852                                        Decanoic acid
1858                                       1-Tetradecanol
1860       Ethylenediaminetetraacetic acid, disodium salt
1862               Hexadecanoic acid, 1-methylethyl ester
Name: Chemical_Name, Length: 180, dtype: str

In [70]:
df_total['Chemical_Name'].to_list()

['Alcohol ethoxylate C12-15/E5 phosphate',
 'C12-13 beta-branched primary alco hol sulfate',
 'C12-13 beta-branched primary alco hol sulfate/1-ethoxylate',
 'N,N-Dimethyl-N-dodecyl amino- betaine',
 'Bis[(1-Methylimidazol)-(2- ethyl- hexanoate)], zinc complex',
 'Alkyl polyglucoside 600',
 'Alcohol ethoxylate C16-18/E14',
 'Alcohol ethoxylate C16-18/E5',
 'Alcohol ethoxylate C12-15/E3',
 'Alcohol ethoxylate C11/E7',
 'Alcohol ethoxylate C11/E3',
 'Tween 80',
 'Tween 80',
 'Propylene glycol',
 'Heptanal',
 'Isopropyl myristate',
 'di-Propylene glycol',
 'Sodium hydroxide',
 'Sodium hydroxide',
 '4-Methylthio benzaldehyde',
 'Heptyl butyrate',
 'Hydroxycitronellal',
 'Methyl caproate',
 'Alkyl dimethyl betaine',
 '1-(2-Isopropylphenyl)-1-phenyle- thane (Mixture of isomers)',
 'Benzyl salicylate',
 'Sodium dodecyl sulphate',
 'Sodium dodecyl sulphate',
 'Sodium dodecyl sulphate',
 'Sodium dodecyl sulphate',
 'Sodium dodecyl sulphate',
 'Sodium carbonate',
 'Benzalkonium chloride',
 '1-Bro

In [71]:
chemical_name = df_total['Chemical_Name'].to_list()
unique_name = set(chemical_name) #set()을 사용하면, 중복된 item은 모두 삭제됩니다.

In [72]:
unique_name

{'(2E)-3,7-Dimethyl-2,6-octadien-1-ol',
 '1,2-Propylene glycol',
 '1-(2-Isopropylphenyl)-1-phenyle- thane (Mixture of isomers)',
 '1-(Spiro[4.5]dec-7-en-7-yl)pent-4- en-1-one (mixture of isomers)',
 '1-Bromo-4-chlorobutane',
 '1-Bromohexane',
 '1-Decanol',
 '1-Dodecanol',
 '1-Hexanol',
 '1-Naphthalene acetic acid',
 '1-Naphthaleneacetic acid',
 '1-Octanol',
 '1-Pentanol',
 '1-Spiro[4.5]dec-7-en-7-yl-4-penten-1-one',
 '1-Tetradecanol',
 '1-tert-Butoxy-2-propanol',
 '10-Undecenoic acid',
 '2-Isopropyl-2-isobutyl-1,3-dime- thoxypropane',
 '3,4-Dimethyl-1H-pyrazole',
 '4-(Methylthio)benzaldehyde',
 '4-Methylthio benzaldehyde',
 'Acetic acid',
 'Alcohol ethoxylate C11/E3',
 'Alcohol ethoxylate C11/E7',
 'Alcohol ethoxylate C12-15/E3',
 'Alcohol ethoxylate C12-15/E5 phosphate',
 'Alcohol ethoxylate C16-18/E14',
 'Alcohol ethoxylate C16-18/E5',
 'Alkyl dimethyl betaine',
 'Alkyl polyglucoside 600',
 'Amines, hydrogenated tallow alkyl',
 'Benzalkonium chloride',
 'Benzene, 1-(1-methylethyl)-2-

In [73]:
len(unique_name) #180개 데이터가 있었지만, 단일 물질 기준으로는 119개가 있네요.

119

In [74]:
for each in unique_name:
    print (each)

Methyl laurate
Hydrochloric acid
4-Methylthio benzaldehyde
Alcohol ethoxylate C11/E3
Bis[(1-Methylimidazol)-(2- ethyl- hexanoate)], zinc complex
Hexyl salicylate
Sodium perborate
Sodium soap
Sodium xylenesulfonate
Methyl palmitate
(2E)-3,7-Dimethyl-2,6-octadien-1-ol
Isopropyl tetradecanoate
Alcohol ethoxylate C12-15/E5 phosphate
Polyethylene glycol
1-Naphthalene acetic acid
Amines, hydrogenated tallow alkyl
Coco alkyltrimethylammonium chlorides
Potassium soap
C12-13 beta-branched primary alco hol sulfate/1-ethoxylate
1-Naphthaleneacetic acid
Alcohol ethoxylate C16-18/E14
Benzalkonium chloride
Linalyl acetate
1-Spiro[4.5]dec-7-en-7-yl-4-penten-1-one
1-Decanol
Benzyl-C8-18-alkyldimethylammonium chlorides
Butyl methacrylate
Carbonic acid sodium salt (1:2)
Water
1,2-Propylene glycol
Hydrogenated tallow amine
Hexadecanoic acid
Methyl hexadecanoate
Ethanol
Ethylenediamine tetraacetic acid disodium salt
Butan-1-ol
Benzyl salicylate
Dodecanol
Propylene glycol
Carbonic acid disodium salt, compd

In [75]:
#label 정보는 동일 화합물이라면 1이거나 0이어야 함. nunique()는 동일 화합물의 label에 있는 정보의 개수 확인.
#groupby(Chemical Name)은 화합물 이름을 기준으로 dataframe을 묶어봅니다.
#이름 기준으로 묶었을 때, label 정보가 몇개 있는지 확인하는 방식.
df_total.groupby('Chemical_Name')['label'].nunique()

Chemical_Name
(2E)-3,7-Dimethyl-2,6-octadien-1-ol                                 1
1,2-Propylene glycol                                                1
1-(2-Isopropylphenyl)-1-phenyle- thane (Mixture of isomers)         1
1-(Spiro[4.5]dec-7-en-7-yl)pent-4- en-1-one (mixture of isomers)    1
1-Bromo-4-chlorobutane                                              1
                                                                   ..
alpha-Terpineol                                                     1
alpha-Terpinyl acetate                                              1
di-Propylene glycol                                                 1
di-n-Propyl disulfide                                               1
n-Pentanol                                                          1
Name: label, Length: 119, dtype: int64

# Quiz2
df_total.groupby('Chemical_Name')['label'].nunique()를 실행했을 때, 도대체 왜 최대값은 2, 최소값은 1이 나올까요?

각 화합물(Chemical_Name)별로 고유한(unique) 라벨(label)이 몇 종류나 있는지 개수를 세는 코드
    이 결과의 최소값이 1, 최대값이 2가 나오는 이유
        
1. 최소값이 1이 나오는 이유 (정상 데이터) 특정 화합물이 데이터셋에 단 1번만 등장했거나, 여러 번 중복해서 등장했더라도 기록된 라벨이 모두 똑같은 경우
예: 화합물 A가 3번 중복 등장했지만 3개의 행 모두 라벨이 Active인 경우, 고유한 라벨의 종류는 Active 1개뿐이므로 nunique() 값은 1
정보가 일치하는 정상적인 상태

2. 최대값이 2가 나오는 이유 (충돌/오류 데이터) 데이터셋의 라벨이 이진 분류(Binary Classification, 예: Active/Inactive 또는 1/0)로 구성되어 있기 때문
예: 화합물 B가 2번 중복 등장했는데, 첫 번째 행에는 Active로 적혀있고 두 번째 행에는 Inactive로
동일한 물질인데 결과가 모순되게 기록된 것
이 경우 해당 화합물이 가질 수 있는 고유한 라벨 종류는 Active와 Inactive 2개이므로 nunique() 값은 최대 2

In [76]:
df_total.groupby('Chemical_Name')['label'].nunique().max()

np.int64(2)

In [77]:
df_total.groupby('Chemical_Name')['label'].nunique().min()

np.int64(1)

In [78]:
label_counts = df_total.groupby('Chemical_Name')['label'].nunique()
bad_names = label_counts[label_counts > 1].index
print (bad_names)
print ('label 정보가 불일치하는 화합물 개수: ',len(bad_names))

Index(['10-Undecenoic acid', 'Acetic acid', 'Alcohol ethoxylate C11/E3',
       'Alcohol ethoxylate C12-15/E5 phosphate', 'Alkyl polyglucoside 600',
       'Benzyl alcohol', 'Dodecanoic acid', 'Ethanol', 'Eugenol', 'Hexanol',
       'Hydrochloric acid', 'Linalyl acetate', 'Octanol', 'Sodium perborate',
       'Tween 80', 'Water'],
      dtype='str', name='Chemical_Name')
label 정보가 불일치하는 화합물 개수:  16


In [79]:
df_filtered = df_total.groupby('Chemical_Name').filter(lambda x: x['label'].nunique() == 1)

print(f"전체: {df_total.shape}행")
print(f"label이 불일치 하는 화합물 제거: {df_filtered.shape}행")

전체: (180, 26)행
label이 불일치 하는 화합물 제거: (143, 26)행


In [80]:
df_cleaned = df_filtered.drop_duplicates(subset='Chemical_Name')
print(f"label이 일치 하는 화합물 중 중복된 결과 제거: {df_cleaned.shape}행")

label이 일치 하는 화합물 중 중복된 결과 제거: (103, 26)행


# Quiz3
1. smiles 코드가 없는 물질 이름을 확인해봅시다.
2. pubchem 데이터베이스 (https://pubchem.ncbi.nlm.nih.gov/)에서 물질 명을 입력해서 검색해봅시다.
3. 구글에서 물질 명을 입력해서 검색해봅시다.
4. CASRN (카스넘버)는 물질의 이름을 표시하는 또 다른 방법입니다. 동일 화학물질이 여러개의 이름을 가질 수 있어서, 고유한 번호를 부여해서 화합물을 찾아요.
5. df_cleaned에서 smiles 코드가 없는 물질들은 왜 없는지 검색을 통해 찾아봅시다.

*사슬 길이가 정해져있지 않아서.
구조가 정해져있지 않다.
코드가 없는 것을 버릴 것인가. 뭐라도 찾아서 넣을 것인가. 양/질

[SMILES 코드가 누락되는 주요 원인 4가지 및 예시]
1. 혼합물 및 천연 유래 성분 탄소 사슬 길이가 하나로 고정되지 않았거나 여러 물질이 섞인 오일, 추출물, 계면활성제 등은 단일 분자 구조로 특정할 수 없어 SMILES 코드가 없다.
C12-13 beta-branched primary alco hol sulfate (탄소가 12개~13개 섞여 있음)
Alcohol ethoxylate C16-18/E14 (탄소가 16개~18개 섞여 있음)
Coco-betaine, Cocotrimethyl ammonium chloride (코코넛 유래 혼합물)
Hydrogenated tallow amine, Fatty acids, potassium salts (지방산 혼합물)
3. 고분자 화합물 (Polymers) 분자량이 매우 크고 수많은 반복 단위가 섞여 있어, 명확한 단일 SMILES 코드로 정의하기 어렵다.
Polysorbate 80 (대표적인 고분자 비이온성 계면활성제)
4. 무기물, 염(Salts) 및 금속 착화합물 SMILES는 주로 탄소 기반의 유기 화합물 표기에 최적화되어 있어, 복잡한 금속 이온 결합이나 염 형태의 물질은 데이터베이스에 명확히 등록되지 않는 경우가 많다.
Bis[(1-Methylimidazol)-(2- ethyl- hexanoate)], zinc complex (아연 착화합물)
Potassium soap, Sodium soap (칼륨/나트륨 비누염)
Benzalkonium chloride (염화물 형태이면서 동시에 알킬 사슬 길이가 혼합된 물질)
5. 모호한 명칭 (Ambiguous names) 단일 화학 구조를 지칭하지 않고 'Butanols'처럼 여러 이성질체를 포함하는 복수형이거나 포괄적인 그룹명으로 라벨링된 경우, 정확히 어떤 분자인지 하나로 매핑할 수 없다.
Butanols (1-부탄올, 2-부탄올 등 여러 이성질체가 존재하여 특정 불가)
[데이터 정제 결론] 이러한 물질들은 컴퓨터가 단일 분자 구조의 특징(Feature)을 읽어내고 학습할 수 없게 만드는 노이즈(Noise) 데이터다. 따라서 머신러닝 모델의 정확도를 높이기 위해 데이터 셋 정제 과정에서 해당 물질들을 모두 제거하는 것이 타당하다.

In [81]:
#smiles 코드가 없는 물질?
df_cleaned['SMILES']

1                                                     NaN
2                                                     NaN
5                                                     NaN
6                                                     NaN
8                                                     NaN
                              ...                        
1757                               COCC(COC)(CC(C)C)C(C)C
1764                          C=CCCC(=O)C1=CCCC2(CCCC2)C1
1858                                      CCCCCCCCCCCCCCO
1860    [Na+].[Na+].OC(=O)CN(CCN(CC(O)=O)CC([O-])=O)CC...
1862                           CCCCCCCCCCCCCCCC(=O)OC(C)C
Name: SMILES, Length: 103, dtype: str

In [82]:
df_cleaned['SMILES'].isna()

1        True
2        True
5        True
6        True
8        True
        ...  
1757    False
1764    False
1858    False
1860    False
1862    False
Name: SMILES, Length: 103, dtype: bool

In [83]:
df_smi = df_cleaned.dropna(subset=['SMILES'])

In [84]:
df_smi.shape

(81, 26)

In [85]:
df_smi['label'].value_counts()

label
0    56
1    25
Name: count, dtype: int64

In [86]:
missing_smiles_names = df_cleaned[df_cleaned['SMILES'].isna()]['Chemical_Name'].unique()
print("SMILES가 없는 물질 목록:", missing_smiles_names)

SMILES가 없는 물질 목록: <ArrowStringArray>
[              'C12-13 beta-branched primary alco hol sulfate',
  'C12-13 beta-branched primary alco hol sulfate/1-ethoxylate',
                       'N,N-Dimethyl-N-dodecyl amino- betaine',
 'Bis[(1-Methylimidazol)-(2- ethyl- hexanoate)], zinc complex',
                               'Alcohol ethoxylate C16-18/E14',
                                'Alcohol ethoxylate C16-18/E5',
                                'Alcohol ethoxylate C12-15/E3',
                                   'Alcohol ethoxylate C11/E7',
                                      'Alkyl dimethyl betaine',
                                       'Benzalkonium chloride',
                                              'Potassium soap',
                                                 'Sodium soap',
                                   'Hydrogenated tallow amine',
                                                  'Butan-1-ol',
                             'Cocotrimethyl ammonium chloride',
   

In [87]:
df_smi.to_csv('skin_irritation_cleaned.csv',index=False)